# A4 — Ultralytics YOLO Experiment Notebook

This notebook is the **primary interface** of an A4 project (see the repository `README.md`).
It walks through the full lifecycle of a fine-tuning experiment — setup, dataset
validation, training, evaluation, summary, export, and report — while delegating all
core ML functionality to **Ultralytics** (package + Platform).

Beyond "run these cells," each section below also explains *why* the step exists and
*what can go wrong* if it's skipped. In practice, most failed YOLO projects fail for
one of two reasons: a dataset problem that was never caught before training started, or
an experiment that can't be reproduced because nobody recorded what was actually run.
This notebook is structured specifically to make both mistakes hard to make.

**How to use this template**
- Copy this notebook into `projects/<Task>/<Dataset>/notebook.ipynb`.
- Fill in the marked configuration cells (task, model, dataset, experiment name).
- Put dataset-specific YAML under `configs/datasets/`, and any *intentional* overrides
  under `configs/experiments/<experiment_name>.yaml`.
- Run cells top to bottom. Custom code should stay rare and minimal — if Ultralytics or
  Ultralytics Platform already does it, use it (see the repository's `yolo-*` skills).
- All Ultralytics-generated outputs live under `runs/` — this template does not
  duplicate them into a separate `artifacts/`, `checkpoints/`, or `logs/` structure.

## 0. Environment

Every notebook session (especially on Colab/Kaggle, where the VM is thrown away between
sessions) starts from a blank environment. This section makes sure three things are true
before anything else runs: the right package version is installed, the runtime can
actually see a usable accelerator, and dataset paths will resolve where you expect them
to instead of triggering a surprise re-download.

In [ ]:
%pip install -q -U ultralytics

In [ ]:
!git clone https://github.com/AmirMahdiRezaeiEECS/ALL-IN-ONE-VISION-4
%cd ALL-IN-ONE-VISION-4

Colab/Kaggle images ship with a fixed `ultralytics` version that can lag months behind
the current release. Reinstalling with `-U` at the start of every session costs a few
seconds and avoids chasing bugs that were already fixed upstream.

In [ ]:
import os
from pathlib import Path

import yaml

from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
print(f"Working directory: {PROJECT_ROOT}")

`yolo checks` prints installed versions (torch, CUDA, ultralytics), whether a GPU is
visible, and free disk space. Run it once per session and actually read the output —
training "succeeding" silently on CPU because no GPU was detected is a common way to
waste hours waiting for what should have taken minutes.

In [ ]:
!yolo checks

**Optional — Ultralytics Platform.** Platform (not a bespoke tracker) is this repo's
primary tool for dataset/experiment/model management — see the README's design
principles. Setting an API key unlocks two things: `data=ul://username/datasets/dataset-slug`
(train against a Platform-hosted dataset without manually downloading it) and
`project=username/project-slug` (mirror this run's metrics into a Platform project you
can browse and compare visually later). Neither is required — everything below works
identically against a purely local `data.yaml` and a local `runs/` directory.

The `datasets_dir` setting matters for a more mundane reason: `data.yaml` files often
declare a *relative* `path:`, and Ultralytics resolves that relative path against
whatever `datasets_dir` currently points to. If that setting is stale from a previous
session, you'll see `"Dataset not found, attempting download"` even though your data is
sitting right there on disk — Ultralytics just resolved the path to the wrong place.
Setting `datasets_dir` explicitly once per environment avoids that entire class of bug.

In [ ]:
# os.environ["ULTRALYTICS_API_KEY"] = "..."  # uncomment to enable Platform streaming

# If `path:` in your data.yaml is relative, it resolves against `datasets_dir` below.
# Point it at your actual dataset root once per environment (Colab/Kaggle sessions reset).
# !yolo settings datasets_dir=/content/datasets

## 1. Experiment Setup

An **experiment**, for our purposes, is one reproducible combination of a task, a model,
a dataset, and a set of hyperparameter overrides — everything below in one place, so
that six months from now (or a teammate reading this notebook) can tell exactly what
was run without archaeology.

Two design choices are worth understanding, not just following:

- **Task selection matters architecturally, not just semantically.** `TASK` isn't a
  label for humans — it determines which model head, loss function, and label format
  Ultralytics expects. A `detect` head predicts `(x, y, w, h, class)`; a `-seg` head also
  predicts a mask; a `-pose` head also predicts keypoints. Picking the wrong `TASK` for a
  given checkpoint doesn't error loudly — it trains "successfully" against the wrong
  target and produces a model that never learns anything useful. That's exactly why the
  sanity-check cell below exists.
- **Config-driven overrides encode a philosophy, not just tidiness.** Ultralytics ships
  defaults that were themselves tuned across many experiments; overriding a default
  should be a deliberate, recorded decision ("we raised `imgsz` because objects are
  small"), not an accumulation of settings someone once tried and forgot to remove.
  Keeping overrides in a small YAML file next to the notebook makes every deviation from
  the baseline explicit and diffable across experiments.

In [ ]:
# --- Experiment identity -----------------------------------------------------
TASK = "detect"  # detect | segment | semantic | depth | classify | pose | obb
MODEL = "yolo26n.pt"  # ALWAYS a pretrained checkpoint; start with 'n' to validate the
                        # pipeline cheaply, then scale up (see the yolo-models skill)
DATA = "configs/datasets/dataset.yaml"  # local yaml, classify folder, or ul://... URI

EXPERIMENT_NAME = "baseline"  # self-describing, e.g. "0906_yolo26n_voc_e100"
PROJECT = f"runs/{TASK}"       # or "username/project-slug" to stream to Platform

# --- Config-driven overrides ---------------------------------------------------
# Only settings we have a strong reason to change belong here (README: "Default-First
# Configuration"). Everything else is left to Ultralytics.
experiment_config_path = Path(f"configs/experiments/{EXPERIMENT_NAME}.yaml")
overrides = {}
if experiment_config_path.exists():
    with open(experiment_config_path) as f:
        overrides = yaml.safe_load(f) or {}

print(f"Task:      {TASK}")
print(f"Model:     {MODEL}")
print(f"Data:      {DATA}")
print(f"Run:       {PROJECT}/{EXPERIMENT_NAME}")
print(f"Overrides: {overrides}")

**Why this check exists.** Model filenames encode the task in a suffix
(`-seg`, `-pose`, `-obb`, ...; no suffix means plain detection). If that suffix doesn't
match `TASK`, Ultralytics will still happily build a model and run epochs — there's no
hard error, because from the framework's point of view you asked for a valid
combination, just an internally inconsistent one. The failure shows up later, indirectly,
as **mAP stuck near zero** — a symptom that looks identical to "the model can't learn
this data" and can send you hunting for a data problem that doesn't exist. This cell
catches the mismatch immediately, when it's a one-line fix, instead of after a full
training run. It only warns (rather than blocking) because open-vocabulary and
promptable families like YOLO-World, YOLOE, SAM, or RT-DETR don't follow this naming
convention at all.

In [ ]:
_TASK_SUFFIX = {
    "detect": "", "segment": "-seg", "semantic": "-sem", "depth": "-depth",
    "classify": "-cls", "pose": "-pose", "obb": "-obb",
}
_stem = Path(MODEL).stem
_all_suffixes = [s for s in _TASK_SUFFIX.values() if s]
_expected = _TASK_SUFFIX.get(TASK, "")

if _stem.startswith(("yolo",)):  # only applies to plain YOLO family naming
    if _expected and not _stem.endswith(_expected):
        print(f"WARNING: MODEL='{MODEL}' does not end in '{_expected}' for TASK='{TASK}'.")
    elif not _expected and any(_stem.endswith(s) for s in _all_suffixes):
        print(f"WARNING: MODEL='{MODEL}' looks task-suffixed but TASK='{TASK}' (detect) was chosen.")

## 2. Dataset Preparation & Validation

**Garbage in, garbage out — but slower and more expensive than in most ML problems**,
because a bad YOLO dataset usually doesn't crash; it trains for hours and produces a
model that quietly doesn't work. The four checks in this section are ordered from
cheapest/fastest to most expensive/most conclusive, so problems get caught as early as
possible:

1. Structural validation (does the YAML even parse, do the paths exist) — milliseconds.
2. A single visual spot check (is *this one* label actually on the object) — seconds.
3. A real one-epoch smoke test (does the *entire* dataloader work end to end) — minutes.
4. Reading what that smoke test already produced (is the class balance sane) — free.

Steps 3 and 4 deliberately reuse Ultralytics' own machinery instead of writing custom
dataset-inspection code. If Ultralytics already builds a class histogram and an
augmentation-space plot as a side effect of training, hand-rolling a second version of
that logic in the notebook would just be a second place for it to be subtly wrong.

If raw annotations still need converting (COCO/DOTA/masks), use the built-in converters
in `ultralytics.data.converter` (see the `yolo-datasets` skill); keep any one-off
conversion script under `scripts/data/` rather than inline here.

In [ ]:
from ultralytics.data.utils import check_det_dataset

if TASK != "classify":
    dataset_info = check_det_dataset(DATA)  # validates data.yaml, resolves paths
    names = dataset_info["names"]
    print(f"Classes ({len(names)}): {names}")
else:
    dataset_info = None
    names = None
    print(f"Classification dataset folder: {DATA}")

`check_det_dataset` only checks *structure* — required keys are present, the requested
split's path resolves, and (for known public datasets) an auto-download can proceed. It
does **not** open every image or parse every label file, so a clean result here means
"the dataset is wired up correctly," not "every label is correct." That's what the next
two steps are for.

**Visual spot check** (detect only — five-column `class cx cy w h` rows, where `cx, cy`
is the box **center**, not the top-left corner — a classic source of "boxes offset in
`train_batch*.jpg`"). Ultralytics finds an image's label file by mirroring its path: the
*last* `/images/` segment becomes `/labels/`, and the extension becomes `.txt`. That rule
is implemented once, correctly, in `img2label_paths` — reusing it here (instead of a
hand-written `str.replace("/images", "/labels")`) avoids a subtle bug: a naive replace
rewrites the *first* occurrence of "images" in the path, which silently points at the
wrong file if that substring appears more than once (e.g. a folder literally named
`images_dataset`).

In [ ]:
from ultralytics.data.utils import visualize_image_annotations
from ultralytics.data.utils import img2label_paths

if TASK == "detect":
    train_dir = dataset_info["train"]
    train_dir = Path(train_dir[0] if isinstance(train_dir, list) else train_dir)
    sample_image = next(
        (p for ext in ("*.jpg", "*.jpeg", "*.png") for p in train_dir.glob(ext)), None
    )
    sample_label = Path(img2label_paths([str(sample_image)])[0])
    visualize_image_annotations(str(sample_image), str(sample_label), label_map=names)

**Task-loader smoke test.** This is the single most reliable dataset check available,
because it doesn't inspect the dataset from the outside — it builds the *actual*
PyTorch dataset and dataloader Ultralytics will use for real training, on a small
(`fraction=0.1`), cheap (`epochs=1`) slice. Anything that would break real training
(missing files, malformed labels, a `kpt_shape` mismatch, wrong polygon format for a
`-seg` model, ...) breaks here too, in under a minute instead of after a multi-hour run.
Set `RUN_SMOKE_TEST = True` once per new dataset or after any label changes.

In [ ]:
RUN_SMOKE_TEST = False
SMOKE_DIR = Path(PROJECT) / "smoke_test"

if RUN_SMOKE_TEST:
    YOLO(MODEL).train(
        data=DATA,
        epochs=1,
        fraction=0.1,
        project=PROJECT,
        name="smoke_test",
        exist_ok=True,
    )

These three plots are generated automatically by the smoke test above — no custom
analysis code needed:

- **`labels.jpg`** — a class-frequency histogram plus the distribution of box
  center positions, widths, and heights. A wildly imbalanced histogram tells you now,
  before spending compute, that rare classes will need oversampling or more data.
- **`labels_correlogram.jpg`** — pairwise correlations between box parameters. Useful
  mainly for spotting systematic labeling artifacts (e.g. every box the same size,
  which usually means a labeling-tool default was never adjusted).
- **`train_batch0.jpg`** — actual augmented training images with their labels drawn on
  top. This is the fastest way a human will ever catch a labeling error: if boxes look
  wrong here, no amount of training will fix it, because the model is being shown the
  wrong ground truth.

In [ ]:
from IPython.display import Image, display

if RUN_SMOKE_TEST:
    # Class balance, aug-space coverage, and label correctness — all from Ultralytics,
    # none of it hand-rolled.
    for plot_name in ("labels.jpg", "labels_correlogram.jpg", "train_batch0.jpg"):
        plot_path = SMOKE_DIR / plot_name
        if plot_path.exists():
            display(Image(filename=str(plot_path)))

## 3. Model & Training

**Why start from a pretrained checkpoint at all?** `yolo26n.pt` was trained on millions
of images (COCO and friends) and already encodes general-purpose visual features — edges,
textures, shapes, common object parts. Fine-tuning reuses those features and only
adapts the model to your specific classes, which is why a few dozen epochs on a few
thousand images can work at all. Training the same architecture from random weights
(`pretrained=False`) throws that knowledge away and typically needs on the order of
100× more data to reach comparable accuracy — which is why the README calls this out
explicitly as a mistake to avoid, and why this notebook never exposes
`pretrained=False` as an option.

Class-count mismatches between the pretrained checkpoint and your dataset are handled
automatically: the classification head is re-initialized for your class count while the
backbone (the feature-extracting part) keeps its pretrained weights, so a 3-class
`data.yaml` on an 80-class checkpoint just works without any manual surgery.

In [ ]:
model = YOLO(MODEL)  # always start from pretrained weights

model.train(
    data=DATA,
    project=PROJECT,
    name=EXPERIMENT_NAME,
    **overrides,
)

RUN_DIR = Path(model.trainer.save_dir)
print(f"Run saved to: {RUN_DIR}")

Every run gets its own numbered directory under `runs/` (`baseline`, `baseline2`, ...) —
nothing here overwrites a previous experiment, which is what makes side-by-side
comparison possible later. `RUN_DIR` captures exactly which directory this run landed
in, since Ultralytics decides the final name (it may not be `EXPERIMENT_NAME` verbatim
if that name was already taken).

**Optional — hyperparameter tuning.** Ultralytics' built-in tuner is a genetic
algorithm: it trains many short, cheap runs, each with slightly mutated hyperparameters
(learning rate, augmentation strengths, loss weights, ...), keeps the mutations that
improved validation fitness, and repeats. It is a real, useful tool — and also the
*least* effective lever available, typically worth only 0.5–2 mAP, because it can only
rearrange the model's response to the data you already have. Per the `yolo-tuning`
skill's improvement playbook, every cheaper lever should be exhausted first: fix
labeling issues visible in the confusion matrix, train longer if validation mAP was
still rising, increase `imgsz` if objects are small, move up a model size if both train
and val metrics are mediocre (underfitting), and only then tune. Running the genetic
search before that is a common way to spend a lot of GPU time optimizing hyperparameters
for a dataset or model choice that was the actual problem. Leave `RUN_TUNE = False` for
a first pass.

In [ ]:
RUN_TUNE = False

if RUN_TUNE:
    tune_model = YOLO(MODEL)
    tune_model.tune(
        data=DATA,
        epochs=30,
        iterations=100,
        plots=False,
        save=False,
        val=False,
    )
    # Inspect runs/<task>/tune/best_hyperparameters.yaml, then retrain fully with it:
    # overrides.update(yaml.safe_load(open("runs/<task>/tune/best_hyperparameters.yaml")))

## 4. Evaluation & Inspection

**Always evaluate `best.pt`, never `last.pt`.** During training, Ultralytics checkpoints
after every epoch and separately tracks which epoch had the best validation fitness so
far. `last.pt` is simply whatever the most recent epoch produced — if the model started
overfitting near the end of training, `last.pt` can be meaningfully worse than the
checkpoint from ten epochs earlier. `best.pt` is that better checkpoint; `last.pt`'s only
real purpose is resuming an interrupted run.

`val()` here uses a very permissive confidence threshold (`conf=0.001` by default) —
much lower than the `0.25` typically used at inference. That's intentional: evaluation
wants to see *every* prediction the model is capable of making, across the whole
confidence range, in order to compute a precision-recall curve. Picking a single
deployment `conf` threshold happens later, using that curve (see the F1 plot below) —
conflating the two would understate the model's real capability.

In [ ]:
BEST = RUN_DIR / "weights" / "best.pt"
best_model = YOLO(BEST)

metrics = best_model.val(data=DATA, plots=True, save_json=True)
print(metrics.results_dict)

**How to read each plot** — five minutes here catches problems that headline metrics
hide:

- **`results.png`** — per-epoch train/val loss and val mAP curves. Train loss still
  falling while val mAP has flattened (or started dropping) is **overfitting**: the
  model is memorizing training data rather than generalizing, and the fix is more/better
  data or a smaller model, not more epochs. Both curves flat and mediocre is
  **underfitting**: the model hasn't learned enough, and the fix is a bigger model, more
  epochs, or higher `imgsz` — not tuning.
- **`confusion_matrix.png`** — rows are predicted class, columns are true class. A
  strong off-diagonal cluster between two specific classes usually means either those
  classes are genuinely visually similar or your labeling was inconsistent between them.
  A heavy background *row* means real objects are being missed (low recall, needs more
  or better examples of that class); a heavy background *column* means the model is
  hallucinating detections on background (needs background/negative examples, or a
  higher inference `conf`).
- **`PR_curve.png`** — precision vs. recall as the confidence threshold sweeps from 0 to
  1. The area under this curve *is* mAP; a curve that collapses early tells you the
  model runs out of confident, correct predictions quickly, even if it can find most
  objects at low confidence.
- **`F1_curve.png`** — this is the practical tool for choosing your deployment `conf`
  threshold: it's precision and recall combined into one number, plotted against
  confidence, so the peak marks the threshold that best balances the two for your data.
- **`labels.jpg`** — shown again here post-training for convenience; the same class-
  balance plot from Section 2.

In [ ]:
for plot_name in (
    "labels.jpg",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
):
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path)))

**Why look at raw predictions when you already have mAP?** A single aggregate number
can hide a model that's excellent on 90% of cases and consistently fails on a specific
subset — a lighting condition, an object orientation, a rare class — that never shows up
as a red flag in `results.png`. Skimming actual predicted images is the cheapest way to
catch failure modes that a scalar metric was never going to surface.

In [ ]:
import cv2
from PIL import Image as PILImage

val_dir = dataset_info["val"] if dataset_info else DATA
val_dir = val_dir[0] if isinstance(val_dir, list) else val_dir

predictions = best_model.predict(source=val_dir, conf=0.25, verbose=False)
for r in predictions[:5]:
    display(PILImage.fromarray(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)))

## 5. Experiment Summary

This is the scientific-method step: an experiment is only useful for comparison if its
configuration and its result are recorded *together*, in one place. Six months from now,
"we tried a bigger model and it helped" is a much weaker statement than a table row
showing exactly which config produced which mAP. Everything printed here already exists
on disk (`args.yaml`, `results.csv`) — this cell doesn't compute anything new, it just
assembles what Ultralytics already recorded into one readable summary, and it reads
from `RUN_DIR` rather than only from in-memory variables, so it still works correctly
even if you come back and re-run just this cell later in a fresh session.

In [ ]:
import pandas as pd

run_args = yaml.safe_load(open(RUN_DIR / "args.yaml"))
results_csv = pd.read_csv(RUN_DIR / "results.csv")
results_csv.columns = [c.strip() for c in results_csv.columns]

if "metrics" not in dir():
    best_model = YOLO(RUN_DIR / "weights" / "best.pt")
    metrics = best_model.val(data=DATA)

summary = {
    "experiment": EXPERIMENT_NAME,
    "task": TASK,
    "model": MODEL,
    "data": DATA,
    "epochs_ran": int(results_csv["epoch"].iloc[-1]) + 1,
    "overrides": overrides,
    "final_metrics": metrics.results_dict,
    "run_dir": str(RUN_DIR),
}

for key, value in summary.items():
    print(f"{key}: {value}")

## 6. Export

Training and deployment usually run on different hardware, and `.pt` (a PyTorch
checkpoint) isn't the right format for most of it — mobile phones, browsers, edge
devices, and many inference servers need a format built for that specific runtime.
Exporting **converts** the model's computation graph into that target format; which
format to pick is really a question about where the model will run
(NVIDIA GPU → `engine`/TensorRT, Apple devices → `coreml`, cross-platform/unsure →
`onnx`), not a matter of taste. The full 20-format matrix with per-format supported
arguments lives in the `yolo-export` skill.

`quantize` controls numeric precision: `16` (FP16) roughly halves size and speeds up
inference on supported hardware with minimal accuracy loss; `8` (INT8) goes further but
needs representative calibration data and can cost noticeably more accuracy on some
models. There's no universally correct choice — it's an accuracy/speed/size trade-off
that should be benchmarked on the actual deployment target, not assumed.

In [ ]:
EXPORT_FORMAT = "onnx"  # torchscript | onnx | openvino | engine | coreml | ...

export_path = best_model.export(format=EXPORT_FORMAT)
print(f"Exported to: {export_path}")

**Why re-validate after exporting?** Format conversion is a real transformation of the
model, not a lossless repackaging — different numeric precision, different operator
implementations, and (if configured incorrectly) different preprocessing can each
silently shift accuracy. Running the same `val()` metric against both the original
checkpoint and the exported artifact is the only way to confirm the conversion didn't
quietly break something before it reaches production, where a regression is much more
expensive to catch.

In [ ]:
exported_model = YOLO(export_path)
exported_metrics = exported_model.val(data=DATA).results_dict

print("Baseline (.pt):", summary["final_metrics"])
print("Exported:      ", exported_metrics)

## 7. Report

The report is the experiment's paper trail: what was configured, what happened, and
what came out of it, written down in one durable file instead of living only in this
notebook's cell outputs (which don't survive a kernel restart) or in someone's memory.
This is what makes an experiment reproducible by someone other than the person who ran
it — including a future version of you.

In [ ]:
report_path = Path("docs") / f"{EXPERIMENT_NAME}_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

report = f"""# Experiment Report — {EXPERIMENT_NAME}

## Configuration
- Task: {TASK}
- Model: {MODEL}
- Data: {DATA}
- Overrides: {overrides}

## Training
- Run directory: `{RUN_DIR}`
- Epochs ran: {summary['epochs_ran']}

## Evaluation
- Final metrics: {summary['final_metrics']}

## Export
- Format: {EXPORT_FORMAT}
- Artifact: `{export_path}`
- Exported metrics: {exported_metrics}

## Notes
_Add qualitative observations, failure cases, and next steps here._
"""

report_path.write_text(report)
print(f"Report written to {report_path}")

---
For details on any stage beyond this notebook's scope, consult the matching skill:
`yolo-models`, `yolo-datasets`, `yolo-training`, `yolo-tuning`, `yolo-inference`, `yolo-export`.